In [ ]:
from functools import partial
from methylbert.data.finetune_data_generate import finetune_data_generate
from methylbert.data.nanopore.finetune_extract import ont_read_extract

# sc_dataset file lists BOTH run-1 BAMs with their cell-type labels, tab-sep:
#   /tmp/bauerste/colo829/PAU59949....bam      T
#   /tmp/bauerste/colo829bl/PAU59807....bam    N

finetune_data_generate(
    f_dmr="dmrs.tsv",
    output_dir="/tmp/bauerste/finetune_run1",
    f_ref="/home/bauerste/GRCh38_no_alt_analysis_set/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna",
    sc_dataset="run1_bams.tsv",
    n_mers=3,
    n_dmrs=50,                       # top-50 per ctype -> 100 DMRs, selected here
    split_ratio=0.85,                # 85% train / 15% eval, split BY READ NAME
    ignore_sex_chromo=False,         # see caution 2 (keeps chrX)
    methyl_caller="dorado",
    read_extract_sequences_func=partial(ont_read_extract, mm_flag="?"),
)

In [ ]:
from methylbert.data.vocab import MethylVocab
from methylbert.data.dataset import MethylBertFinetuneDataset

v = MethylVocab(k=3)
ds = MethylBertFinetuneDataset("/tmp/bauerste/finetune_run1/train_seq.csv", vocab=v, seq_len=510, n_cores=4)
print("rows:", len(ds))
print("num_dmrs (embedding table size):", ds.num_dmrs())   # MUST be 100, not ~35000
print("ctype label counts:", ds.ctype_label_count)

item = ds[0]
d, m = item["dna_seq"], item["methyl_seq"]
print("dna len:", d.shape[0], " methyl len:", m.shape[0])   # both MUST be 512
print("first token (expect SOS id 3):", int(d[0]))
nonpad = (d != v.pad_index).nonzero().flatten()
print("last non-pad token (expect EOS id 2):", int(d[nonpad[-1]]))
print("methyl alphabet:", set(m.tolist()))                  # subset of {0,1,2,3}
print("ctype_label:", item["ctype_label"])                  # 0 or 1